# Phase 2 on a TPU v5e — `fused_gated_mlp`

The first hardware run in this repo. Four things are being measured, in the order
they stop mattering if the runtime dies:

1. **Correctness on device.** Interpret mode is not proof of hardware correctness
   ([jax#36287](https://github.com/jax-ml/jax/issues/36287)), so nothing here is
   publishable until cell 2 is green.
2. **The VMEM budget a `pallas_call` actually gets** — the default of
   `--xla_tpu_scoped_vmem_limit_kib`, which is not published anywhere and which
   `pltpu.InterpretParams` cannot be asked, because it models no VMEM capacity.
3. **Where the kernel sits against the roofline**, with XLA alongside.
4. **Whether copy elision is contractual on the hardware path**, from a DMA count
   in an xprof trace.

Every cell is independently re-runnable and writes its artifact before the next
one starts. No cell should take more than about three minutes.

## The predictions, registered before the run

The byte model is the corrected one. The grid is
`(tokens // block_t, hidden_dim // block_h)` with the hidden axis **innermost**,
and the `w_*` `index_map`s vary in the inner index — so one `t` step walks every
hidden block, and when the inner index resets, block 0 is no longer the
previously fetched slice. Copy elision skips only *consecutive* identical
slices, so the weights are re-read **once per `t` step**:

```
bytes = (tokens // block_t) · 3·E·H·dtype  +  2·T·E·dtype
```

At the tested geometry (`T=256, block_t=128`) that is 193.5 MB and intensity
**63**, not 97.9 MB and 125. Both counts are in `shapes.mlp_bytes`
(`elide_weights=True` gives the one-pass counterfactual); cell 6 adjudicates.

v5e publishes one compute roof, for bf16, and TPU emulates an fp32 matmul with
3 or 6 bf16 passes. So there are three candidate roofs, three ridges, and two
crossovers that a timing sweep can actually resolve:

| `precision` | passes | peak | ridge | HBM-bound at | at the ridge | compute-bound at |
|---|---|---|---|---|---|---|
| `DEFAULT` | 1 | 197 TFLOP/s | 240.5 | 128, 256 | **512** (+1.4%) | 1024, 2048 |
| `HIGH` | 3 | 65.7 TFLOP/s | 80.2 | **128** (−21%) | — | 256, 512, 1024, 2048 |
| `HIGHEST` | 6 | 32.8 TFLOP/s | 40.1 | — | — | every T |

`DEFAULT` at T=512 is registered **in advance as unresolvable**: 1.4% is inside
the noise of any timing collected here, so it is not scored either way. `HIGH`'s
crossover between T=128 and T=256 is the better test of the byte model, because
its margin is 21%.

If the timings do not bend at those two crossovers, the byte model is wrong —
which is the interesting result, not a failure.

**Second prediction:** the on-device error against `reference.gated_mlp` will be
far larger than the CPU run's 3.28e-6 at `DEFAULT`, because the multiplies are
bf16. `GEMMA_RTOL = 1e-4` is a CPU measurement and is expected to fail here. The
notebook **reports** the error rather than asserting an inherited tolerance.

## 0 · Preflight

Refuses to go on without a TPU, and prints the hardware constants everything
below is measured against. `tpu_info` is the only primary source found that
states v5e VMEM; `docs.cloud.google.com/tpu/docs/v5e` does not.

In [ ]:
import jax

from jax._src.pallas.mosaic import tpu_info as ti

print("jax        ", jax.__version__)
print("devices    ", jax.devices())

platform = jax.devices()[0].platform
assert platform == "tpu", (
    f"backend is {platform!r}, not tpu. Runtime > Change runtime type > TPU. "
    "No number produced here would be publishable."
)

# The first argument must be the ChipVersion enum member; a string fails late
# and unhelpfully on `.is_lite`. Second argument is TensorCores per device.
info = ti.get_tpu_info_for_chip(ti.ChipVersion.TPU_V5E, 1)
for field in (
    "bf16_ops_per_second",
    "mem_bw_bytes_per_second",
    "vmem_capacity_bytes",
    "smem_capacity_bytes",
    "hbm_capacity_bytes",
    "num_mxus",
    "mxu_column_size",
    "num_lanes",
    "num_sublanes",
):
    print(f"{field:26} {getattr(info, field):>20,}")

print()
print(f"VMEM {info.vmem_capacity_bytes / 1024**2:.0f} MiB is the *physical* capacity.")
print("What a pallas_call is allowed to use is the scoped-vmem limit -- cell 4.")

## 1 · Setup

Clone and install the package only. **Do not `pip install jax`** — the Colab TPU
runtime ships its own build and reinstalling it breaks `libtpu`.

In [ ]:
import pathlib
import sys

REPO = pathlib.Path("/content/gemma3-tpu-pallas")
if not REPO.exists():
    !git clone --depth 1 https://github.com/denis-mil/gemma3-tpu-pallas {REPO}

%pip install -q -e {REPO} --no-deps

# Belt and braces: makes the cell re-runnable without a kernel restart if the
# editable install has not yet landed on sys.path.
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

RESULTS = pathlib.Path("/content/results")
TRACES = pathlib.Path("/content/traces")
RESULTS.mkdir(exist_ok=True)
TRACES.mkdir(exist_ok=True)

import gemma3_pallas

print("package at", gemma3_pallas.__file__)

## 2 · Correctness on device

The load-bearing cell. `interpret=False`, Gemma's real geometry, against
`reference.gated_mlp` — and it **reports** the error instead of asserting the
CPU-derived tolerance, because `DEFAULT` precision means bf16 multiplies and
1e-4 is not expected to survive that.

A timing number produced without this cell having run is not publishable. That
is the ADR-0003 contract.

In [ ]:
import json

import numpy as np
import jax.numpy as jnp

from gemma3_pallas import bench
from gemma3_pallas.mlp import fused_gated_mlp
from gemma3_pallas.reference import gated_mlp
from gemma3_pallas.shapes import GEMMA3_1B

bench.require_tpu()

TOKENS, BLOCK_T, BLOCK_H = 256, 128, 768

x, w_gate, w_up, w_down = bench.operands(TOKENS)
reference_out = np.asarray(jax.jit(gated_mlp)(x, w_gate, w_up, w_down))
scale = float(np.abs(reference_out).max())


def error_report(precision, *, block_t=BLOCK_T, block_h=BLOCK_H):
    """Max absolute and relative error against the fp32 reference, on device."""
    got = np.asarray(
        fused_gated_mlp(
            x,
            w_gate,
            w_up,
            w_down,
            block_t=block_t,
            block_h=block_h,
            interpret=False,
            precision=precision,
        )
    )
    absolute = np.abs(got - reference_out)
    # Guarded: the output has near-zero entries, and a relative error against
    # one of those says more about the divisor than about the kernel.
    denominator = np.maximum(np.abs(reference_out), 1e-3 * scale)
    return {
        "precision": bench.precision_label(precision),
        "tokens": TOKENS,
        "block_t": block_t,
        "block_h": block_h,
        "output_scale": scale,
        "max_abs_error": float(absolute.max()),
        "max_rel_error": float((absolute / denominator).max()),
        "cpu_reference_max_abs_error": 3.28e-6,
    }


correctness = [error_report(None)]
(RESULTS / "correctness.json").write_text(json.dumps(correctness, indent=2))

row = correctness[0]
print(f"output scale        {row['output_scale']:.4g}")
print(f"max abs error       {row['max_abs_error']:.4g}   (CPU interpret run: 3.28e-6)")
print(f"max rel error       {row['max_rel_error']:.4g}")
print()
print("Nonzero but bounded is the expected shape of this result. A max abs error")
print("of ~1e-2 at DEFAULT is bf16 multiplies, not a bug; 0.0 would be the")
print("surprise, and a NaN or a value comparable to the output scale is a bug.")

## 3 · Precision probe

The same check at all three precisions. There is no published fp32 peak for
v5e — TPU emulates an fp32 matmul with 3 or 6 bf16 passes — so the roof the
kernel is measured against depends on which precision it ran at. The error
should fall roughly monotonically; the pass count whose error approaches the CPU
run's 3.28e-6 identifies the roof to use in cell 7.

In [ ]:
for precision in ("high", "highest"):
    correctness.append(error_report(precision))

(RESULTS / "correctness.json").write_text(json.dumps(correctness, indent=2))

print(f"{'precision':<10} {'max abs':>12} {'max rel':>12}   vs CPU 3.28e-6")
for row in correctness:
    ratio = row["max_abs_error"] / 3.28e-6
    print(
        f"{row['precision']:<10} {row['max_abs_error']:>12.4g} "
        f"{row['max_rel_error']:>12.4g}   {ratio:>10.1f}x"
    )

print()
print("Read this as: whichever precision brings the error down to the CPU number")
print("is the one doing full-mantissa work, and its pass count is the divisor on")
print("the 197 TFLOP/s roof. Set PASSES in cell 7 from this table.")

## 4 · VMEM sweep — the budget a `pallas_call` actually gets

The other load-bearing cell. `pltpu.InterpretParams` models **no** VMEM capacity,
so this question cannot be asked anywhere but here.

The method: run configs of growing working set at the default limit and record
which ones compile. A Mosaic VMEM failure is the measurement, not an error, so
`bench_kernel` records it with the compiler's message and the sweep continues.
The largest working set that compiles and the smallest that does not **bracket**
the default. Then the first failing config is re-run with explicit
`vmem_limit_bytes` values to confirm the flag moves that boundary — which also
confirms the budget is sweepable per `pallas_call`, with no runtime restart.

In [ ]:
cfg = GEMMA3_1B


def working_set_bytes(block_t, block_h, *, cfg=cfg, dtype_bytes=4, buffers=2):
    """VMEM the block choice asks for, per grid step.

    Five blocked operands -- x, w_gate, w_up, w_down and the output -- each
    double-buffered, plus the two [block_t, block_h] fp32 intermediates that
    live inside the kernel body and are not buffered at all. Those two are what
    a 22.5 MiB estimate leaves out; at 128x768 they are 0.375 MiB each.
    """
    blocked = (
        block_t * cfg.embed_dim  # x
        + 2 * cfg.embed_dim * block_h  # w_gate, w_up
        + block_h * cfg.embed_dim  # w_down
        + block_t * cfg.embed_dim  # output
    )
    intermediates = 2 * block_t * block_h  # gate, up
    return dtype_bytes * (buffers * blocked + intermediates)


BLOCK_HS = [128, 256, 384, 768, 1152, 1728, 3456, 6912]
BLOCK_TS = [128, 256]

vmem_specs = [
    dict(tokens=256, block_t=bt, block_h=bh, precision=None)
    for bt in BLOCK_TS
    for bh in BLOCK_HS
]
vmem_specs.sort(key=lambda s: working_set_bytes(s["block_t"], s["block_h"]))

vmem_results = []
vmem_path = RESULTS / "vmem_sweep.json"


def keep_vmem(result):
    vmem_results.append(result)
    bench.save(vmem_results, vmem_path)  # rewritten in full after every config
    mib = working_set_bytes(result.block_t, result.block_h) / 1024**2
    median = result.median()
    timing = f"{1e3 * median:8.3f} ms" if median else "        --"
    print(f"block_t={result.block_t:<4} block_h={result.block_h:<5} "
          f"{mib:7.2f} MiB  {result.status:<7} {timing}")


bench.sweep(vmem_specs, interpret=False, on_result=keep_vmem)

ok = [r for r in vmem_results if r.status == "ok"]
failed = [r for r in vmem_results if r.status == "failed"]

print()
if ok:
    largest = max(ok, key=lambda r: working_set_bytes(r.block_t, r.block_h))
    print(f"largest that compiled:  {working_set_bytes(largest.block_t, largest.block_h) / 1024**2:.2f} MiB "
          f"(block_t={largest.block_t}, block_h={largest.block_h})")
if failed:
    smallest = min(failed, key=lambda r: working_set_bytes(r.block_t, r.block_h))
    print(f"smallest that failed:   {working_set_bytes(smallest.block_t, smallest.block_h) / 1024**2:.2f} MiB "
          f"(block_t={smallest.block_t}, block_h={smallest.block_h})")
    print()
    print("first failure, verbatim:")
    print(smallest.error[:800])
else:
    print("Nothing failed: the sweep did not reach the budget. The bound is only")
    print("that the default exceeds the largest working set above -- say that,")
    print("rather than reporting the largest config as the limit.")

### 4b · Does `vmem_limit_bytes` move the boundary?

If cell 4 found a failing config, re-run it with explicit limits. A config that
fails at the default and succeeds at a larger explicit limit proves two things at
once: the flag governs the budget, and the default is below the value that
worked. If nothing failed, this cell walks the limit *down* instead, until the
headline geometry stops compiling — which brackets the default from the other
side.

In [ ]:
MiB = 1024**2
LIMITS = [16 * MiB, 32 * MiB, 64 * MiB, 100 * MiB, 128 * MiB]

if failed:
    smallest = min(failed, key=lambda r: working_set_bytes(r.block_t, r.block_h))
    probe_t, probe_h = smallest.block_t, smallest.block_h
    print(f"probing the first failing config: block_t={probe_t}, block_h={probe_h}, "
          f"{working_set_bytes(probe_t, probe_h) / MiB:.2f} MiB\n")
else:
    probe_t, probe_h = BLOCK_T, BLOCK_H
    print(f"nothing failed at the default; walking the limit down on the headline "
          f"geometry: block_t={probe_t}, block_h={probe_h}, "
          f"{working_set_bytes(probe_t, probe_h) / MiB:.2f} MiB\n")

limit_specs = [
    dict(
        tokens=256,
        block_t=probe_t,
        block_h=probe_h,
        precision=None,
        vmem_limit_bytes=limit,
    )
    for limit in LIMITS
]


def keep_limit(result):
    vmem_results.append(result)
    bench.save(vmem_results, vmem_path)
    print(f"vmem_limit={result.vmem_limit_bytes / MiB:6.0f} MiB  {result.status}")


bench.sweep(limit_specs, interpret=False, on_result=keep_limit)

## 5 · Token sweep — the roofline points

`T ∈ {128, 256, 512, 1024, 2048}`, kernel and `jax.jit(reference.gated_mlp)` at
each. `block_t = min(T, best_block_t)`, so the smaller sequences are one `t` step
and read the weights once — which is the geometry the registered intensity table
was computed at.

Warmup 3, repeats 20, `block_until_ready` on every call, and the whole
distribution kept rather than a mean: a bimodal set of samples is a fact about
the run.

In [ ]:
BEST_BLOCK_T = BLOCK_T
BEST_BLOCK_H = BLOCK_H  # set these from cell 4 if a larger block compiled

TOKEN_SWEEP = [128, 256, 512, 1024, 2048]
SWEEP_PRECISIONS = [None, "high"]  # DEFAULT and HIGH: one resolvable crossover each

token_specs = []
for precision in SWEEP_PRECISIONS:
    for tokens in TOKEN_SWEEP:
        token_specs.append(
            dict(
                tokens=tokens,
                block_t=min(tokens, BEST_BLOCK_T),
                block_h=BEST_BLOCK_H,
                precision=precision,
            )
        )
        token_specs.append(dict(label="xla", tokens=tokens, precision=precision))

token_results = []
token_path = RESULTS / "token_sweep.json"


def keep_token(result):
    token_results.append(result)
    bench.save(token_results, token_path)
    median = result.median()
    timing = f"{1e3 * median:8.3f} ms" if median else "        --"
    print(f"{result.label:<7} T={result.tokens:<5} {result.precision:<8} "
          f"{result.status:<7} {timing}")


bench.sweep(token_specs, interpret=False, on_result=keep_token)

## 6 · Profiler — the copy-elision measurement

Trace about five iterations at the headline geometry and count the `w_*` DMAs.
The two byte models disagree by exactly a factor of `tokens // block_t`:

* weights re-read per `t` step → `tokens // block_t` transfers per weight
* copy elision across the inner-index reset → 1 transfer per weight

At `T=256, block_t=128` that is 2 versus 1, and the trace settles it. This is
also the highest-value single use of TPU time on the standing question of whether
copy elision is contractual on the hardware path — the docs state it as a
property, but the `pallas_call` pipeline on TPU is emitted by Mosaic, out of
reach from Python.

In [ ]:
import jax.profiler

from gemma3_pallas.shapes import arithmetic_intensity, mlp_bytes, mlp_flops


def headline(x, w_gate, w_up, w_down):
    return fused_gated_mlp(
        x, w_gate, w_up, w_down,
        block_t=BEST_BLOCK_T, block_h=BEST_BLOCK_H, interpret=False, precision=None,
    )


traced = jax.jit(headline)
jax.block_until_ready(traced(x, w_gate, w_up, w_down))  # compile outside the trace

with jax.profiler.trace(str(TRACES)):
    for _ in range(5):
        jax.block_until_ready(traced(x, w_gate, w_up, w_down))

passes = TOKENS // BEST_BLOCK_T
per_weight = cfg.embed_dim * cfg.hidden_dim * 4
print(f"trace written to {TRACES}")
print()
print(f"grid ({TOKENS // BEST_BLOCK_T}, {cfg.hidden_dim // BEST_BLOCK_H}), "
      f"{cfg.hidden_dim // BEST_BLOCK_H} hidden blocks per t step")
print(f"predicted w_* DMAs per weight, per call: {passes} "
      f"(elision would give 1)")
print(f"total weight bytes, {passes}-pass model: "
      f"{3 * passes * per_weight / 1e6:.1f} MB")
print(f"total weight bytes, elided model:        {3 * per_weight / 1e6:.1f} MB")
print()
for elide in (False, True):
    moved = mlp_bytes(TOKENS, block_t=BEST_BLOCK_T, elide_weights=elide)
    intensity = arithmetic_intensity(mlp_flops(TOKENS), moved)
    print(f"elide_weights={str(elide):<5} bytes={moved / 1e6:7.1f} MB  I={intensity:6.2f}")
print()
print("Count the w_gate/w_up/w_down HBM-to-VMEM transfers per call in the trace")
print("and compare. Re-open it locally with tensorboard-plugin-profile.")

## 7 · Summary and roofline plot

`PASSES` selects the compute roof and has no default anywhere in the library —
inventing one is exactly what `roofline_bound`'s required keyword exists to
prevent. Set it from cell 3.

A point within 2% of its ridge prints as **at ridge**, not as a verdict. That is
not a hedge added after seeing the data: `DEFAULT` at T=512 was registered as
unresolvable before the run.

In [ ]:
import matplotlib.pyplot as plt

from gemma3_pallas.shapes import V5E

PASSES = 1  # 1 = DEFAULT, 3 = HIGH, 6 = HIGHEST -- read cell 3 before setting

default_rows = [r for r in token_results if r.precision == "DEFAULT"]
high_rows = [r for r in token_results if r.precision == "HIGH"]

print(bench.summarise(default_rows, passes=1))
print()
print(bench.summarise(high_rows, passes=3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

intensities = [2**i for i in range(0, 12)]
for passes, name in ((1, "DEFAULT (1 bf16 pass)"), (3, "HIGH (3)"), (6, "HIGHEST (6)")):
    peak = V5E.peak_flops(passes)
    roof = [min(peak, i * V5E.peak_hbm_bandwidth) / 1e12 for i in intensities]
    ax.plot(intensities, roof, label=f"{name} — ridge {V5E.ridge_point(passes):.0f}")

for rows, passes, marker in ((default_rows, 1, "o"), (high_rows, 3, "s")):
    xs, ys = [], []
    for r in rows:
        if r.status != "ok" or r.block_t is None:
            continue
        flops = mlp_flops(r.tokens)
        xs.append(arithmetic_intensity(flops, mlp_bytes(r.tokens, block_t=r.block_t)))
        ys.append(flops / r.median() / 1e12)
    if xs:
        ax.scatter(xs, ys, marker=marker, zorder=3, label=f"kernel, {passes} pass(es)")

ax.set_xscale("log", base=2)
ax.set_yscale("log", base=2)
ax.set_xlabel("arithmetic intensity (FLOP/byte)")
ax.set_ylabel("achieved (TFLOP/s)")
ax.set_title("fused_gated_mlp on TPU v5e — three roofs, one kernel")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS / "roofline.png", dpi=150)
plt.show()

## 8 · Download

`results/` and `traces/` are gitignored; the numbers reach the repo as prose in
`docs/measurements/0001-fused-gated-mlp-on-v5e.md`. Pull the raw artifacts down
so the trace can be re-opened locally against the byte models.

In [ ]:
import shutil

staging = pathlib.Path("/content/phase2_v5e")
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RESULTS, staging / "results")
shutil.copytree(TRACES, staging / "traces")

archive = shutil.make_archive("/content/phase2_v5e", "zip", root_dir=staging)
print(archive, f"{pathlib.Path(archive).stat().st_size / 1e6:.1f} MB")
for path in sorted(staging.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(staging), f"{path.stat().st_size / 1e3:.0f} kB")

from google.colab import files

files.download(archive)